<a href="https://www.kaggle.com/code/princelv84/coco-2017-dataset?scriptVersionId=321528473" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# EchoLens: Fine-tuning on MS-COCO

Fine-tunes the pretrained EchoLens model (Flickr30k) on MS-COCO 2017.

**Kaggle inputs needed:**
- Dataset: `coco-2017-dataset`
- Model: your Flickr30k output added as a Kaggle Model (needs `caption_model.keras` + `vectorization_layer_state.pkl`)

In [ ]:
# Cell 1 — Install
!pip install pycocotools -q

In [ ]:
# Cell 2 — Imports
import os
import re
import json
import pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization

# Force TF2 Keras (avoids any Keras 3 routing on environments that have both)
print(f"TensorFlow : {tf.__version__}")
print(f"Keras      : {keras.__version__}")
print(f"NumPy      : {np.__version__}")

AUTOTUNE = tf.data.AUTOTUNE
tf.random.set_seed(42)

In [ ]:
# Cell 3 — Paths
COCO_TRAIN_IMAGES = "/kaggle/input/coco-2017-dataset/coco2017/train2017"
COCO_VAL_IMAGES   = "/kaggle/input/coco-2017-dataset/coco2017/val2017"
COCO_TRAIN_ANNOT  = "/kaggle/input/coco-2017-dataset/coco2017/annotations/captions_train2017.json"
COCO_VAL_ANNOT    = "/kaggle/input/coco-2017-dataset/coco2017/annotations/captions_val2017.json"

# Your Flickr30k model added as Kaggle Model input
PRETRAINED_MODEL_PATH = "/kaggle/input/models/princelv84/echolens/tensorflow2/default/1/caption_model.keras"
PRETRAINED_VOCAB_PATH = "/kaggle/input/models/princelv84/echolens/tensorflow2/default/1/vectorization_layer_state.pkl"

OUTPUT_DIR      = "/kaggle/working"
FINETUNED_MODEL = f"{OUTPUT_DIR}/caption_model_coco.keras"
FINETUNED_VOCAB = f"{OUTPUT_DIR}/vectorization_layer_state.pkl"

for p in [PRETRAINED_MODEL_PATH, PRETRAINED_VOCAB_PATH, COCO_TRAIN_ANNOT, COCO_VAL_ANNOT]:
    print(f"  {'✓' if os.path.exists(p) else '✗ NOT FOUND'}  {p}")

In [ ]:
# Cell 4 — Config (must match Flickr30k training exactly)
config = {
    "IMAGE_SIZE"      : (299, 299),
    "VOCAB_SIZE"      : 10000,
    "SEQ_LENGTH"      : 25,
    "EMBED_DIM"       : 512,
    "FF_DIM"          : 512,
    "NUM_HEADS"       : 2,        # must match training
    "BATCH_SIZE"      : 32,
    "FINETUNE_EPOCHS" : 10,
    "FINETUNE_LR"     : 1e-5,
    "UNFREEZE_LAST_N" : 20,
    "MAX_COCO_TRAIN"  : None,
    "MAX_COCO_VAL"    : None,
}
print("Config:", config)

In [ ]:
# Cell 5 — Load COCO annotations
def load_coco_captions(annot_path, images_dir, max_images=None):
    with open(annot_path, "r") as f:
        data = json.load(f)
    id_to_file      = {img["id"]: img["file_name"] for img in data["images"]}
    caption_mapping = {}
    text_data       = []
    seen_images     = set()
    for ann in data["annotations"]:
        img_id   = ann["image_id"]
        caption  = ann["caption"].strip().lower()
        filename = id_to_file.get(img_id)
        if not filename:
            continue
        img_path = os.path.join(images_dir, filename)
        if not os.path.exists(img_path):
            continue
        tokens = caption.split()
        if len(tokens) < 5 or len(tokens) > config["SEQ_LENGTH"]:
            continue
        caption_with_tokens = "<start> " + caption + " <end>"
        text_data.append(caption_with_tokens)
        if img_path not in caption_mapping:
            caption_mapping[img_path] = []
            seen_images.add(img_path)
        caption_mapping[img_path].append(caption_with_tokens)
        if max_images and len(seen_images) >= max_images:
            break
    print(f"  Loaded {len(caption_mapping)} images, {len(text_data)} captions")
    return caption_mapping, text_data

print("Loading COCO train annotations...")
train_coco, _ = load_coco_captions(COCO_TRAIN_ANNOT, COCO_TRAIN_IMAGES, max_images=config["MAX_COCO_TRAIN"])
print("Loading COCO val annotations...")
val_coco,   _ = load_coco_captions(COCO_VAL_ANNOT,   COCO_VAL_IMAGES,   max_images=config["MAX_COCO_VAL"])
print(f"Train: {len(train_coco)} images  |  Val: {len(val_coco)} images")

In [ ]:
# Cell 6 — Vocabulary (reuse Flickr30k vocab with numpy compat fix)
#
# FIX 1: NumpyCompatUnpickler
# pkl was saved with NumPy 2.x ('numpy._core'), Kaggle has 1.23.5 ('numpy.core').
# The shim rewrites the module path at read-time — no reinstalls needed.

strip_chars = r"!\"#$%&'()*+,-./:;<=>?@[\]^_`{|}~".replace("<", "").replace(">", "")

def custom_standardization(input_string):
    lowercase = tf.strings.lower(input_string)
    return tf.strings.regex_replace(lowercase, "[%s]" % re.escape(strip_chars), "")

vectorization = TextVectorization(
    max_tokens=config["VOCAB_SIZE"],
    output_mode="int",
    output_sequence_length=config["SEQ_LENGTH"],
    standardize=custom_standardization,
)

class NumpyCompatUnpickler(pickle.Unpickler):
    """Remaps numpy._core → numpy.core for NumPy 1.x / 2.x cross-compatibility."""
    def find_class(self, module, name):
        if module.startswith("numpy._core"):
            module = module.replace("numpy._core", "numpy.core")
        return super().find_class(module, name)

print(f"NumPy version: {np.__version__}")
print("Loading Flickr30k vocabulary (numpy-compat shim)...")
with open(PRETRAINED_VOCAB_PATH, "rb") as f:
    vocab_data = NumpyCompatUnpickler(f).load()

vectorization.set_vocabulary(vocab_data["vocab"])
vocab        = vectorization.get_vocabulary()
index_lookup = dict(zip(range(len(vocab)), vocab))
print(f"Vocabulary loaded: {len(vocab)} tokens ✓")

# Re-save in protocol=4 (numpy-version-agnostic)
with open(FINETUNED_VOCAB, "wb") as f:
    pickle.dump({"config": vectorization.get_config(), "vocab": vocab}, f, protocol=4)
print(f"Vocab saved (protocol=4) → {FINETUNED_VOCAB} ✓")

In [ ]:
# Cell 7 — Dataset pipeline (fixed for variable COCO captions per image)

NUM_CAPTIONS_PER_IMAGE = 5   # pad/truncate all images to exactly this many captions

def decode_and_resize(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, config["IMAGE_SIZE"])
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img

def process_input(img_path, captions):
    return decode_and_resize(img_path), vectorization(captions)

image_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomContrast(0.2),
    layers.RandomBrightness(factor=0.1),
])

def pad_captions(captions, n=NUM_CAPTIONS_PER_IMAGE):
    """Pad or truncate caption list to exactly n entries — makes it rectangular."""
    captions = captions[:n]                          # truncate if too many
    while len(captions) < n:                         # pad by repeating last
        captions = captions + [captions[-1]]
    return captions

def make_dataset(caption_mapping, shuffle=True):
    images   = list(caption_mapping.keys())
    # Pad/truncate every image to exactly NUM_CAPTIONS_PER_IMAGE captions
    captions = [pad_captions(v, NUM_CAPTIONS_PER_IMAGE)
                for v in caption_mapping.values()]

    ds = tf.data.Dataset.from_tensor_slices((images, captions))   # now rectangular ✓
    if shuffle:
        ds = ds.shuffle(config["BATCH_SIZE"] * 8)
    ds = ds.map(process_input, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(config["BATCH_SIZE"]).prefetch(AUTOTUNE)
    return ds

print("Building datasets...")
train_dataset = make_dataset(train_coco, shuffle=True)
valid_dataset = make_dataset(val_coco,   shuffle=False)
print(f"Train batches : {len(train_dataset)}  |  Val batches : {len(valid_dataset)}")

In [ ]:
# Cell 8 — Define model architecture
#
# FIX 2: Rebuild from scratch instead of loading .keras
# The .keras file was saved with Keras 3 (keras.saving.serialize_keras_object),
# but Kaggle runs TF-bundled Keras 2. Loading the file calls the LEGACY
# deserializer which doesn't know 'Functional' → ValueError.
#
# Solution: rebuild the identical architecture here, then load ONLY the weights
# using model.load_weights() with by_name=True. Weights transfer perfectly
# because layer names are identical.

# ── Transformer Encoder Block ────────────────────────────────────────────────
class TransformerEncoderBlock(layers.Layer):
    def __init__(self, embed_dim, dense_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim   = embed_dim
        self.dense_dim   = dense_dim
        self.num_heads   = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, dropout=0.0)
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.dense_1     = layers.Dense(embed_dim, activation="relu")

    def call(self, inputs, training, mask=None):
        inputs = self.layernorm_1(inputs)
        inputs = self.dense_1(inputs)
        attn   = self.attention_1(
            query=inputs, value=inputs, key=inputs, training=training)
        return self.layernorm_2(inputs + attn)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"embed_dim": self.embed_dim,
                    "dense_dim": self.dense_dim,
                    "num_heads": self.num_heads})
        return cfg


# ── Positional Embedding ─────────────────────────────────────────────────────
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings    = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(
            input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length = sequence_length
        self.vocab_size      = vocab_size
        self.embed_dim       = embed_dim
        self.embed_scale     = tf.math.sqrt(tf.cast(embed_dim, tf.float32))

    def call(self, inputs):
        length    = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        return (self.token_embeddings(inputs) * self.embed_scale
                + self.position_embeddings(positions))

    def compute_mask(self, inputs, mask=None):
        return tf.math.not_equal(inputs, 0)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"sequence_length": self.sequence_length,
                    "vocab_size": self.vocab_size,
                    "embed_dim": self.embed_dim})
        return cfg


# ── Transformer Decoder Block ────────────────────────────────────────────────
class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, ff_dim, num_heads, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim   = embed_dim
        self.ff_dim      = ff_dim
        self.num_heads   = num_heads
        self.attention_1 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.attention_2 = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=embed_dim, dropout=0.1)
        self.ffn_layer_1 = layers.Dense(ff_dim,    activation="relu")
        self.ffn_layer_2 = layers.Dense(embed_dim)
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.embedding   = PositionalEmbedding(
            embed_dim=config["EMBED_DIM"],
            sequence_length=config["SEQ_LENGTH"],
            vocab_size=config["VOCAB_SIZE"])
        self.out         = layers.Dense(config["VOCAB_SIZE"], activation="softmax")
        self.dropout_1   = layers.Dropout(0.3)
        self.dropout_2   = layers.Dropout(0.5)
        self.supports_masking = True

    def call(self, inputs, encoder_outputs, training, mask=None):
        inputs      = self.embedding(inputs)
        causal_mask = self.get_causal_attention_mask(inputs)
        if mask is not None:
            padding_mask  = tf.cast(mask[:, :, tf.newaxis], dtype=tf.int32)
            combined_mask = tf.cast(mask[:, tf.newaxis, :], dtype=tf.int32)
            combined_mask = tf.minimum(combined_mask, causal_mask)
        attn1 = self.attention_1(
            query=inputs, value=inputs, key=inputs,
            attention_mask=combined_mask, training=training)
        out1  = self.layernorm_1(inputs + attn1)
        attn2 = self.attention_2(
            query=out1, value=encoder_outputs, key=encoder_outputs,
            attention_mask=padding_mask, training=training)
        out2    = self.layernorm_2(out1 + attn2)
        ffn_out = self.ffn_layer_1(out2)
        ffn_out = self.dropout_1(ffn_out, training=training)
        ffn_out = self.ffn_layer_2(ffn_out)
        ffn_out = self.layernorm_3(ffn_out + out2, training=training)
        ffn_out = self.dropout_2(ffn_out, training=training)
        return self.out(ffn_out)

    def get_causal_attention_mask(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size, seq = input_shape[0], input_shape[1]
        i    = tf.range(seq)[:, tf.newaxis]
        j    = tf.range(seq)
        mask = tf.cast(i >= j, dtype="int32")
        mask = tf.reshape(mask, (1, seq, seq))
        mult = tf.concat(
            [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)],
            axis=0)
        return tf.tile(mask, mult)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"embed_dim": self.embed_dim,
                    "ff_dim": self.ff_dim,
                    "num_heads": self.num_heads})
        return cfg


# ── LR Schedule ──────────────────────────────────────────────────────────────
class LRSchedule(keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, post_warmup_learning_rate, warmup_steps):
        super().__init__()
        self.post_warmup_learning_rate = post_warmup_learning_rate
        self.warmup_steps = warmup_steps

    def __call__(self, step):
        global_step  = tf.cast(step, tf.float32)
        warmup_steps = tf.cast(self.warmup_steps, tf.float32)
        warmup_lr    = self.post_warmup_learning_rate * (global_step / warmup_steps)
        return tf.cond(global_step < warmup_steps,
                       lambda: warmup_lr,
                       lambda: self.post_warmup_learning_rate)

    def get_config(self):
        return {"post_warmup_learning_rate": float(self.post_warmup_learning_rate),
                "warmup_steps": int(self.warmup_steps)}


# ── ImageCaptioningModel ─────────────────────────────────────────────────────
class ImageCaptioningModel(keras.Model):
    def __init__(self, cnn_model, encoder, decoder,
                 num_captions_per_image=5, image_aug=None, **kwargs):
        super().__init__(**kwargs)
        self.cnn_model   = cnn_model
        self.encoder     = encoder
        self.decoder     = decoder
        self.loss_tracker = keras.metrics.Mean(name="loss")
        self.acc_tracker  = keras.metrics.Mean(name="accuracy")
        self.num_captions_per_image = num_captions_per_image
        self.image_aug   = image_aug

    def calculate_loss(self, y_true, y_pred, mask):
        loss = self.loss(y_true, y_pred)
        mask = tf.cast(mask, dtype=loss.dtype)
        loss *= mask
        return tf.reduce_sum(loss) / tf.reduce_sum(mask)

    def calculate_accuracy(self, y_true, y_pred, mask):
        accuracy = tf.equal(y_true, tf.argmax(y_pred, axis=2))
        accuracy = tf.math.logical_and(mask, accuracy)
        accuracy = tf.cast(accuracy, dtype=tf.float32)
        mask     = tf.cast(mask, dtype=tf.float32)
        return tf.reduce_sum(accuracy) / tf.reduce_sum(mask)

    def _compute_caption_loss_and_acc(self, img_embed, batch_seq, training=True):
        encoder_out    = self.encoder(img_embed, training=training)
        batch_seq_inp  = batch_seq[:, :-1]
        batch_seq_true = batch_seq[:, 1:]
        mask           = tf.math.not_equal(batch_seq_true, 0)
        batch_seq_pred = self.decoder(
            batch_seq_inp, encoder_out, training=training, mask=mask)
        loss = self.calculate_loss(batch_seq_true, batch_seq_pred, mask)
        acc  = self.calculate_accuracy(batch_seq_true, batch_seq_pred, mask)
        return loss, acc

    def train_step(self, batch_data):
        batch_img, batch_seq = batch_data
        batch_loss = 0
        batch_acc  = 0
        if self.image_aug:
            batch_img = self.image_aug(batch_img)
        img_embed = self.cnn_model(batch_img)
        for i in range(self.num_captions_per_image):
            with tf.GradientTape() as tape:
                loss, acc = self._compute_caption_loss_and_acc(
                    img_embed, batch_seq[:, i, :], training=True)
                batch_loss += loss
                batch_acc  += acc
            train_vars = (self.cnn_model.trainable_variables
                          + self.encoder.trainable_variables
                          + self.decoder.trainable_variables)
            grads = tape.gradient(loss, train_vars)
            self.optimizer.apply_gradients(zip(grads, train_vars))
        batch_acc /= float(self.num_captions_per_image)
        self.loss_tracker.update_state(batch_loss)
        self.acc_tracker.update_state(batch_acc)
        return {"loss": self.loss_tracker.result(), "acc": self.acc_tracker.result()}

    def test_step(self, batch_data):
        batch_img, batch_seq = batch_data
        batch_loss = 0
        batch_acc  = 0
        img_embed  = self.cnn_model(batch_img)
        for i in range(self.num_captions_per_image):
            loss, acc = self._compute_caption_loss_and_acc(
                img_embed, batch_seq[:, i, :], training=False)
            batch_loss += loss
            batch_acc  += acc
        batch_acc /= float(self.num_captions_per_image)
        self.loss_tracker.update_state(batch_loss)
        self.acc_tracker.update_state(batch_acc)
        return {"loss": self.loss_tracker.result(), "acc": self.acc_tracker.result()}

    @property
    def metrics(self):
        return [self.loss_tracker, self.acc_tracker]


print("Model classes defined ✓")

In [ ]:
# Cell 9 — Build model architecture from scratch
#
# We build the exact same architecture as Flickr30k training, then
# load ONLY the weights from the .keras file. This fully bypasses
# the Keras 2 vs Keras 3 serialization incompatibility.

def get_cnn_model():
    base_model = keras.applications.EfficientNetB0(
        input_shape=(*config["IMAGE_SIZE"], 3),
        include_top=False,
        weights="imagenet",
    )
    base_model.trainable = False
    base_model_out = base_model.output
    base_model_out = layers.Reshape(
        (-1, base_model_out.shape[-1])
    )(base_model_out)
    cnn_model = keras.models.Model(base_model.input, base_model_out)
    return cnn_model

cnn_model = get_cnn_model()
encoder   = TransformerEncoderBlock(
    embed_dim=config["EMBED_DIM"],
    dense_dim=config["FF_DIM"],
    num_heads=config["NUM_HEADS"]
)
decoder   = TransformerDecoderBlock(
    embed_dim=config["EMBED_DIM"],
    ff_dim=config["FF_DIM"],
    num_heads=config["NUM_HEADS"]
)

caption_model = ImageCaptioningModel(
    cnn_model=cnn_model,
    encoder=encoder,
    decoder=decoder,
    image_aug=image_augmentation,
)

# NEW — build by running sub-components individually, no call() needed
print("Building model (warm-up forward pass)...")
for imgs, seqs in train_dataset.take(1):
    # Build CNN + encoder
    img_feat    = caption_model.cnn_model(imgs, training=False)
    encoder_out = caption_model.encoder(img_feat, training=False)
    # Build decoder
    seq_inp  = seqs[:, 0, :-1]                          # (batch, SEQ_LENGTH-1)
    mask     = tf.math.not_equal(seq_inp, 0)
    _        = caption_model.decoder(seq_inp, encoder_out, training=False, mask=mask)
    break

# Count params from sub-components directly (bypasses Keras 2 build-state tracking)
total_params = (
    sum(tf.size(w).numpy() for w in caption_model.cnn_model.weights) +
    sum(tf.size(w).numpy() for w in caption_model.encoder.weights) +
    sum(tf.size(w).numpy() for w in caption_model.decoder.weights)
)
print(f"Model built — total params: {total_params:,}")

In [ ]:
# Cell 10 — Load pretrained weights into sub-components directly
import zipfile, tempfile, h5py

print(f"Loading weights from: {PRETRAINED_MODEL_PATH}")

with tempfile.TemporaryDirectory() as tmpdir:
    # Step 1: unzip the .keras archive
    with zipfile.ZipFile(PRETRAINED_MODEL_PATH, 'r') as zf:
        all_files = zf.namelist()
        print("Archive contents:", all_files)
        weights_files = [n for n in all_files if n.endswith('.h5')]
        print("H5 files found:", weights_files)
        for wf in weights_files:
            zf.extract(wf, tmpdir)

    weights_path = os.path.join(tmpdir, weights_files[0])

    # Step 2: inspect top-level group names in the h5 file
    with h5py.File(weights_path, 'r') as f:
        top_keys = list(f.keys())
        print("Top-level h5 groups:", top_keys)

    # Step 3: load into each sub-model by name, skip mismatches safely
    for submodel, label in [
        (caption_model.cnn_model, "CNN"),
        (caption_model.encoder,   "Encoder"),
        (caption_model.decoder,   "Decoder"),
    ]:
        try:
            submodel.load_weights(weights_path, by_name=True, skip_mismatch=True)
            print(f"  {label} weights loaded ✓")
        except Exception as e:
            print(f"  {label} load_weights warning (non-fatal): {e}")

print("\nPretrained Flickr30k weights transferred ✓")
print("Training from these weights — catastrophic forgetting avoided.")

In [ ]:
# Cell 11 — Unfreeze top CNN layers
cnn = caption_model.cnn_model
cnn.trainable = True
for layer in cnn.layers:
    layer.trainable = False
unfreeze_from = len(cnn.layers) - config["UNFREEZE_LAST_N"]
for layer in cnn.layers[unfreeze_from:]:
    layer.trainable = True

trainable = sum(1 for l in cnn.layers if l.trainable)
frozen    = sum(1 for l in cnn.layers if not l.trainable)
print(f"CNN → trainable layers: {trainable},  frozen: {frozen}")

In [ ]:
# Cell 12 — Compile
cross_entropy = keras.losses.SparseCategoricalCrossentropy(
    from_logits=False, reduction="none")

caption_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=config["FINETUNE_LR"]),
    loss=cross_entropy
)
print(f"Compiled  lr={config['FINETUNE_LR']} ✓")

In [ ]:
# Cell 13 — Callbacks (FIXED: save_weights_only=True for subclassed model)
FINETUNED_WEIGHTS = f"{OUTPUT_DIR}/best_weights"

callbacks = [
    keras.callbacks.ModelCheckpoint(
        FINETUNED_WEIGHTS,
        save_best_only=True,
        monitor="val_loss",
        save_weights_only=True,   # subclassed models can't save as HDF5/keras format
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
]
print("Callbacks ready ✓")

In [ ]:
# Cell 14 — Fine-tune
print(f"Starting COCO fine-tuning for up to {config['FINETUNE_EPOCHS']} epochs...")
history = caption_model.fit(
    train_dataset,
    epochs=config["FINETUNE_EPOCHS"],
    validation_data=valid_dataset,
    callbacks=callbacks
)

In [ ]:
# Cell 15 — Plot training curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"],     label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Fine-tune Loss (COCO)")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history["acc"],     label="Train Acc")
plt.plot(history.history["val_acc"], label="Val Acc")
plt.title("Fine-tune Accuracy (COCO)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16 — Sanity check on 3 random COCO val images
max_decoded = config["SEQ_LENGTH"] - 1
val_images  = list(val_coco.keys())

def generate_caption(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, config["IMAGE_SIZE"])
    img = tf.image.convert_image_dtype(img, tf.float32)
    plt.imshow(img.numpy())
    plt.axis("off")
    plt.show()
    img      = tf.expand_dims(img, 0)
    img_feat = caption_model.cnn_model(img)
    encoded  = caption_model.encoder(img_feat, training=False)
    decoded_caption = "<start> "
    for i in range(max_decoded):
        tokenized = vectorization([decoded_caption])[:, :-1]
        mask      = tf.math.not_equal(tokenized, 0)
        preds     = caption_model.decoder(tokenized, encoded, training=False, mask=mask)
        token_probs      = preds[0, i, :].numpy()
        token_probs[:4]  = 0   # mask pad/unk/start/end
        token_idx = np.argmax(token_probs)
        token     = index_lookup.get(token_idx, "")
        if token in ("<end>", ""):
            break
        decoded_caption += " " + token
    caption = decoded_caption.replace("<start>", "").strip()
    print("Caption:", caption)
    return caption

print("Sanity check on 3 random COCO val images:")
for img_path in np.random.choice(val_images, 3, replace=False):
    generate_caption(img_path)

In [ ]:
# Cell 17 — Export TFLite (FIXED: restore best weights before export)

# Step 1: Restore best weights saved by ModelCheckpoint
print("Restoring best weights from checkpoint...")
caption_model.load_weights(FINETUNED_WEIGHTS)
print("Best weights restored ✓")

# Step 2: Build encoder functional model
print("\nBuilding TFLite-compatible functional models...")
image_input  = keras.Input(shape=(299, 299, 3), name="image_input")
cnn_features = caption_model.cnn_model(image_input)
enc_features = caption_model.encoder(cnn_features, training=False)
encoder_model = keras.Model(
    inputs=image_input,
    outputs=enc_features,
    name="tflite_encoder"
)

# Step 3: Build decoder functional model (mask baked in — app.py needs no mask input)
seq_input     = keras.Input(shape=(None,), dtype=tf.int32,   name="sequence_input")
enc_out_input = keras.Input(shape=(None, config["EMBED_DIM"]),
                             dtype=tf.float32,               name="encoded_image_input")
mask          = layers.Lambda(lambda x: tf.math.not_equal(x, 0))(seq_input)
preds         = caption_model.decoder(
    seq_input, enc_out_input, training=False, mask=mask)
decoder_model = keras.Model(
    inputs=[seq_input, enc_out_input],
    outputs=preds,
    name="tflite_decoder"
)

print(f"Encoder params : {encoder_model.count_params():,}")
print(f"Decoder params : {decoder_model.count_params():,}")

# Step 4: Convert and save
def convert_and_save(model, filename):
    print(f"\n  Converting {filename} ...")
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS,
        tf.lite.OpsSet.SELECT_TF_OPS,
    ]
    tflite_model = converter.convert()
    out_path = f"{OUTPUT_DIR}/{filename}"
    with open(out_path, "wb") as f:
        f.write(tflite_model)
    size_mb = len(tflite_model) / (1024 * 1024)
    print(f"  Saved → {out_path}  ({size_mb:.2f} MB) ✓")

convert_and_save(encoder_model, "echolens_encoder_quantized.tflite")
convert_and_save(decoder_model, "echolens_decoder_quantized.tflite")

# Step 5: Confirm all output files exist
print("\n✅ Fine-tuning complete!")
print("Download these 3 files from the Output panel and replace in your HF Space:\n")
output_files = [
    "echolens_encoder_quantized.tflite",
    "echolens_decoder_quantized.tflite",
    "vectorization_layer_state.pkl",
]
for fname in output_files:
    full_path = f"{OUTPUT_DIR}/{fname}"
    exists    = "✓" if os.path.exists(full_path) else "✗ MISSING"
    size      = os.path.getsize(full_path) / (1024*1024) if os.path.exists(full_path) else 0
    print(f"  {exists}  {full_path}  ({size:.2f} MB)")

print("\nAfter replacing files in HF Space → restart the Space → test with a car image.")